In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

import logging

In [2]:
# Generar una funcion que obtenga el modelo LLM

def get_llm(prefix_model, temperature=0.9):
    """
    Obtiene un modelo de lenguaje (LLM) basado en el modelo especificado.

    Args:
        model (str): El nombre del modelo a utilizar.
        temperature (float, optional): La temperatura para la generación de texto. 
                                        Valores más altos producen respuestas más creativas.
                                        Valores más bajos producen respuestas más deterministas.
                                        Por defecto es 0.9.

    Returns:
        llm: Un objeto de modelo de lenguaje inicializado.
    """
    llm = init_chat_model(
                            model=prefix_model, 
                            # temperature=temperature
                          )
    return llm

In [3]:
# Configuración del modelo de lenguaje
load_dotenv()

prefix_model = os.getenv("PREFIX_MODEL")

# Obtención del modelo
llm = get_llm(prefix_model)


### Fundamentos de RAG

| Estrategia | Caracteristicas | Limitaciones |
|--|--|--|
| Contexto | Incluir todo la información de los documentos en el PROMPT | Ventana del contexto |
| Fine tuning | Reentranar el modelo, atado a la infraestructura | Tiempo de actualización |
| RAG | Particiona el docuemnto en fragmentos y coloca los fragmentos en el PROMPT | Tratamiento de los documentos |


In [ ]:
# Consulta sin tener la informacion de la politica de devolucion de la empresa TiendaYa

PREGUNTA = "¿Cuál es la política de devolución de la empresa TiendaYa para productos en oferta?"

respuesta = llm.invoke(PREGUNTA)

print("Respuesta: " + respuesta.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Respuesta: Como modelo de inteligencia artificial, no tengo acceso a información en tiempo real ni a bases de datos internas de empresas específicas llamadas "TiendaYa" (ya que puede haber varios comercios con ese nombre en diferentes países). 

Para conocer la política de devolución exacta para productos en oferta de **TiendaYa**, te recomiendo hacer lo siguiente:

1. **Revisar su sitio web oficial:** Busca en el pie de página (footer) secciones como *"Términos y condiciones"*, *"Políticas de devolución"* o *"Preguntas frecuentes (FAQ)"*.
2. **Contactar a su servicio al cliente:** Escríbeles a través de su chat en vivo, correo electrónico o WhatsApp oficial si lo tienen disponible.

*Nota general sobre el comercio electrónico:* Muchas tiendas online suelen tener políticas más restrictivas con los productos en oferta o liquidación (por ejemplo, permitiendo solo cambios por talla/defecto y no devoluciones de dinero), por lo que siempre es mejor confirmarlo directamente con ellos antes d

### Aplicando RAG

- 1. Carga y procesamiento del documento. ( INGESTA DE DATOS)
- 2. Realizar el embedding usando modelos de IA. ( INGESTA DE DATOS)
- 3. Uso de base de datos vectorial ( INGESTA DE DATOS)
- 4. Realización de consultas

#### 1. Carga y procesamiento del documento

In [7]:
# Librería pypdf

from pypdf import PdfReader
from langchain_core.documents import Document

ARCHIVO = "data/politica_tiendaya.pdf"

reader = PdfReader(ARCHIVO)

documentos = []

for i, pagina in enumerate(reader.pages):
    
    texto = pagina.extract_text()
    print(f"Página {i+1}: {texto[:100]}...")  # Muestra los primeros 100 caracteres de cada página

    doc = Document(page_content=texto, 
                   metadata={"pagina": i+1})
    
    documentos.append(doc)

print("Número de páginas procesadas: ", len(documentos))


Página 1: TiendaYa — Política de devoluciones y reembolsos
Versión 2.1 · Documento ficticio para uso en el cur...
Página 2: 5. Productos dañados en el envío
Si el producto llega dañado, el cliente debe reportarlo dentro de l...
Página 3: 8. Cómo iniciar una devolución
El cliente debe ingresar a la sección Mis pedidos, seleccionar el ped...
Número de páginas procesadas:  3


In [12]:
# Segmentacion de documentos

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Se va a segmentar el documento en fragmetos con solapamientos

# Dividir el fragmentos de 500 carateres con un solapamiento de 50 caracteres

chunk_size = 500
chunk_overlap = 50

spliter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,  # fragmento de 500 caracteres
    chunk_overlap=chunk_overlap # solapamiento de 50 caracteres
)

fragmentos = spliter.split_documents(documentos)

print(f"{'chunk_size':>10} {'overlap':>8} {'fragmentos':>11}")
print(f"{chunk_size:>10} {chunk_overlap:>8} {len(fragmentos):>11}")

chunk_size  overlap  fragmentos
       500       50           7


#### 2. Realizar el embedding usando modelos de IA. ( INGESTA DE DATOS)

Un modelo de embeddings convierte un texto en un vector de números. Textos con significado similar producen vectores cercanos; la cercanía se mide con la **similitud coseno** (1 = idénticos, 0 = sin relación). Es el mismo principio del Módulo 1, ahora con modelos entrenados para este fin.

| Modelo | Proveedor | Ejecución | Consideraciones | Dimensiones |
|---|---|---|---|--|
| BGE (bge-m3) | BAAI, abierto | Local, vía Ollama | Multilingüe; sin costo; los datos no salen del equipo |1,024|
| text-embedding-3 | OpenAI | API | Alta calidad; costo por token |3,072|
| gemini-embedding-001 | Gemini | API | Google |3,072|

El modelo de embeddings usado al indexar debe ser **el mismo** que se use al consultar: vectores de modelos distintos no son comparables.

In [13]:
import math 

def similitud_coseno(v1, v2):
    """
    SIMILITUD COSENO = mide el angulo entre dos vectores.
    1.0  -> apuntan exactamente igual (muy parecidos)
    0.0  -> no tienen relacion
    Es la metrica que usan los LLMs reales para comparar embeddings.
    """
    producto_punto = sum(a * b for a, b in zip(v1, v2))
    norma1 = math.sqrt(sum(a * a for a in v1))
    norma2 = math.sqrt(sum(b * b for b in v2))
    if norma1 == 0 or norma2 == 0:
        return 0.0
    return producto_punto / (norma1 * norma2)

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

MODEL_EMBEDDINGS = "gemini-embedding-001"  # Modelo de embeddings de Google GenAI

embeddings = GoogleGenerativeAIEmbeddings(model=MODEL_EMBEDDINGS)  # Modelo de embeddings de Google GenAI


In [16]:
PREGUNTA = "¿Cuál es la política de devolución de la empresa TiendaYa para productos en oferta?"

# Generar el vector de la pregunta
vector_pregunta = embeddings.embed_query(PREGUNTA)

print("Vector de la pregunta: ", vector_pregunta[:200], "...")  # Muestra los primeros 200 valores del vector

Vector de la pregunta:  [0.014368168, -0.007729253, -0.011264546, -0.06577767, -0.00772031, 0.038819022, 0.0022101377, 0.025668778, -0.01367702, -0.01004288, 0.004157194, 0.02342531, 0.0075691566, -0.027637, 0.11160647, -0.00017570842, 0.022733022, 0.006427428, -0.009605082, -0.0072372607, 0.026772715, -0.0035063848, -0.042971432, 0.0048942263, -0.0005997484, 0.007583119, 0.03904725, 0.018836562, 0.020094085, -0.02442837, 0.006625093, 0.023131702, -0.0038103557, 0.011864543, -0.017711055, 0.032715432, -0.006817085, 0.0026362133, 0.027824929, 0.02139734, 0.0046447394, 0.008289933, -0.0014704997, -0.003657237, 0.0040316945, -0.0227497, 0.0059817294, -0.0060909935, 0.024398535, 0.023240564, 0.028442932, -0.0267406, 0.024398386, -0.17591634, 0.004030751, 0.0068769385, -0.009629852, -0.014052126, -0.009539522, -0.046172716, -0.021260144, -0.008094576, -0.025930036, -0.037191987, -0.026974564, 0.015597677, -0.018654648, 0.03927002, 0.0070213405, -0.021270074, 0.023679523, -0.009114265, 0.020